# SpecDist — A100 GPU setup

**Local persistent storage · 4-hour guaranteed session**

| Cell | What it does | Time |
|------|-------------|------|
| 0. Bootstrap | One-time: clone → deps → auth → run | ~3 min + model dl (once) |
| 1. Resume | After 4-hour timeout — pipeline skips completed steps | ~1 min |
| 2. Monitor | State + log tail (auto-refresh option) | instant |
| 3. Verify | Check artifacts before session ends | instant |
| 4. Parallel | All losses simultaneously (multi-GPU or CPU) | ~2-3 h |

---

### Quick-start (terminal — replaces Cells 0–2 for GPU sessions)

```bash
# In your SSH terminal — set secrets first, then:
export GITHUB_TOKEN="ghp_..."   export WANDB_API_KEY="..."

bash $HOME/Distill-Spec-Research/gbv-research/deploy/aip_gpu_setup.sh a10_qwen
# or: aip_gpu_setup.sh a100_qwen
```

If the repo isn't cloned yet:
```bash
git clone https://github.com/Rmuk655/Distill-Spec-Research.git \
  $HOME/Distill-Spec-Research
bash $HOME/Distill-Spec-Research/gbv-research/deploy/aip_gpu_setup.sh a10_qwen
```

---

### AIP instance selection

| AIP instance | GPU | VRAM | CONFIG | Use for |
|---|---|---|---|---|
| `g5.12xlarge` | NVIDIA A10G × 4 | 24 GB each | **`a10_qwen`** | Exploration — 8B BF16, ~80 min/loss |
| `p4d.24xlarge` | NVIDIA A100 40GB × 8 | 40 GB each | **`a100_qwen`** | Paper confirmation — full BF16, 2000 steps |
| `p4de.24xlarge` | NVIDIA A100 80GB × 8 | 80 GB each | `a100_qwen` | Same, more headroom |
| CPU instance | — | — | `server_gpt2` | GPT-2 convergence analysis (CPU-only) |

**GPUs per pod:** 1 for a single run · 4 (A10G) or 8 (A100) for parallel loss training  
**Disable CPU jobs on GPU nodes:** Yes ✅  
**Docker image:** `Training Base (CUDA 12.8.1, Ubuntu 24.04, CUDNN 9.18.1.3, NCCL 2.29.7, Python 3.12)`

---

### CONFIG options

| CONFIG | Instance | Teacher | max_train_prompts | Steps | Time/loss | Purpose |
|--------|----------|---------|-------------------|-------|-----------|---------|
| `a10_qwen` | A10G 24 GB | Qwen3-8B **BF16** | 1000 (2 epochs) | 2000 | ~80 min | Exploration — exact teacher, no quant |
| `a100_qwen` | A100 40 GB | Qwen3-8B **BF16** | none (full data) | 2000 | ~17 min | Paper numbers — full diversity |
| `kaggle` | T4 16 GB | Qwen3-8B NF4 | 500 (2 epochs) | 1000 | ~60 min | T4 trend formation |
| `colab` | T4 15 GB | Qwen3-4B BF16 | 250 (2 epochs) | 500 | ~30 min | T4 safe fallback |
| `colab_lite` | any GPU | Qwen3-1.7B BF16 | 100 (3 epochs) | 300 | ~10 min | Quick smoke test |
| `server_gpt2` | CPU only | GPT-2-medium | 200 (5 epochs) | 1000 | ~4-6 hr | CPU convergence proof |

> **Epoch formula:** `epochs = steps / max_train_prompts`. 2 epochs = minimum to see trend with 8B teacher.

**GPU → config:**
- 24 GB VRAM (A10G) → `a10_qwen`  (BF16, exploration quality, free on AIP)
- 40/80 GB VRAM (A100) → `a100_qwen`  (BF16, paper quality)
- No GPU → `server_gpt2`

### Session strategy

AIP sessions guarantee **4 hours**. Always `BACKGROUND = True` + **Cell 1 (Resume)**.

- **`a10_qwen`:** 2000 steps × ~2.5 s/step ≈ 83 min/loss. Run 1 loss/session, or Cell 4 (4 GPUs × 5 losses = all in ~2 h).
- **`a100_qwen`:** 2000 steps × ~0.5 s/step ≈ 17 min/loss. All losses in one 4-hour session.

### W&B

```bash
export WANDB_API_KEY="..."            # https://wandb.ai/authorize
export WANDB_BASE_URL=https://adobesensei.wandb.io   # Adobe enterprise (optional)
```

In [ ]:
# =============================================================================
# Cell 0 — BOOTSTRAP  (run once per fresh AIP session)
#
# GPU sessions: use the terminal quick-start instead (faster):
#   bash .../deploy/aip_gpu_setup.sh a10_qwen
#
# Set secrets in the VS Code terminal BEFORE running this cell:
#   export GITHUB_TOKEN="ghp_..."      # required for private repo
#   export WANDB_API_KEY="..."         # from https://wandb.ai/authorize
#   export HF_TOKEN="hf_..."           # optional (Qwen3 models are public)
#
# After a session timeout use Cell 1 (Resume) instead — it is self-contained.
# =============================================================================

import os

# -- Edit these ---------------------------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"

# Storage: persistent across sessions. Uses $USER (your LDAP) automatically.
_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
STORAGE_HOME = os.path.expanduser("~")

REPO_DIR     = f"{STORAGE_HOME}/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = f"{STORAGE_HOME}/specdist"

# CONFIG — choose based on your AIP instance:
#   a10_qwen    — g5.12xlarge  (A10G  24 GB): 8B BF16, 2000 steps, ~80 min/loss  ← GPU default
#   a100_qwen   — p4d.24xlarge (A100  40 GB): 8B BF16, 2000 steps, ~17 min/loss  ← paper runs
#   server_gpt2 — CPU instance:              GPT-2, 1000 steps, CPU-only
#   colab_lite  — any GPU, quick smoke test: 1.7B, 300 steps, ~10 min
CONFIG       = "a10_qwen"   # ← change to a100_qwen for A100 instance

SMOKE        = True    # Always smoke first! Set False once smoke passes.
BACKGROUND   = True    # ALWAYS True: 4-hour AIP session limit requires background mode
LOSSES       = None    # None = all losses  |  "kl" or "kl,kl_tree" = subset
EXTRA_ARGS   = []
# -----------------------------------------------------------------------------

import subprocess, sys

# Verify local storage is accessible
if not os.path.isdir(STORAGE_HOME):
    print(f"WARNING: local storage path not found: {STORAGE_HOME}")
    print(f"  Check: ls /sensei-fs-3/users/")
    print(f"  Set STORAGE_HOME manually above if your path differs.")
else:
    print(f"local storage  : {STORAGE_HOME}")

gh = os.environ.get("GITHUB_TOKEN", "")
if not gh:
    print("WARNING: GITHUB_TOKEN not set — clone will fail for a private repo.")

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned to local storage")
else:
    if gh:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url],
                       capture_output=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                       capture_output=True, text=True)
    print(r.stdout.strip() or "Already up to date")

# HF model cache → local storage (persists across sessions)
hf_cache = os.path.join(STORAGE_ROOT, "hf_cache")
os.makedirs(hf_cache, exist_ok=True)
os.environ["HF_HOME"]            = hf_cache
os.environ["TRANSFORMERS_CACHE"] = hf_cache
for _flag in ("TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_OFFLINE"):
    os.environ.pop(_flag, None)
print(f"HF cache   : {hf_cache}")
print(f"Config     : {CONFIG}  |  smoke={SMOKE}  |  losses={LOSSES or 'all'}")

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

# GPU check
try:
    import torch
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}      : {p.name}  {p.total_memory/1024**3:.0f} GB")
except Exception:
    print("GPU        : (torch not yet installed — will install in bootstrap)")

bootstrap(CONFIG, STORAGE_ROOT, REPO_DIR, gbv_dir=GBV_DIR,
          warn_vram_below_gb=20.0)

_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR,
                 smoke=SMOKE, losses=LOSSES,
                 background=BACKGROUND, extra_args=EXTRA_ARGS)

In [ ]:
# =============================================================================
# Cell 1 — RESUME  (after 4-hour session expiry or manual restart)
#
# Self-contained: re-clones if needed, re-installs deps, then resumes.
# The pipeline reads pipeline_state_*.json and skips completed steps.
# Local artifacts (checkpoints, results.db, HF cache) are already there.
#
# Re-export secrets in terminal if they expired:
#   export WANDB_API_KEY="..."  &&  export GITHUB_TOKEN="ghp_..."
# =============================================================================

import os, subprocess, sys

# -- Edit these (must match Cell 0) -------------------------------------------
REPO_URL     = "https://github.com/Rmuk655/Distill-Spec-Research.git"
_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
STORAGE_HOME = os.path.expanduser("~")

REPO_DIR     = f"{STORAGE_HOME}/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = f"{STORAGE_HOME}/specdist"
CONFIG       = "a10_qwen"   # must match Cell 0  (a10_qwen | a100_qwen | server_gpt2)
# -----------------------------------------------------------------------------

gh = os.environ.get("GITHUB_TOKEN", "")
clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url],
                       capture_output=True)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
                       capture_output=True, text=True)
    print(r.stdout.strip() or "Already up to date")

# Restore HF cache env var — local storage cache is already populated
hf_cache = os.path.join(STORAGE_ROOT, "hf_cache")
os.environ["HF_HOME"]            = hf_cache
os.environ["TRANSFORMERS_CACHE"] = hf_cache
for _flag in ("TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "HF_HUB_OFFLINE"):
    os.environ.pop(_flag, None)

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

bootstrap(CONFIG, STORAGE_ROOT, REPO_DIR, gbv_dir=GBV_DIR,
          warn_vram_below_gb=20.0)

print(f"\nResuming {CONFIG} — completed steps are skipped automatically.\n")
_ = run_pipeline(CONFIG, STORAGE_ROOT, GBV_DIR, background=True)

In [ ]:
# =============================================================================
# Cell 2 — MONITOR  (safe to run any time, including while pipeline runs)
# Set AUTO_REFRESH = True for a live tail; interrupt the cell to stop.
# =============================================================================

import os, sys

_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
STORAGE_HOME = os.path.expanduser("~")

REPO_DIR     = f"{STORAGE_HOME}/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = f"{STORAGE_HOME}/specdist"
CONFIG       = "a10_qwen"   # must match Cell 0/1  (a10_qwen | a100_qwen | server_gpt2)
AUTO_REFRESH = False         # True = live tail loop (interrupt cell to stop)
REFRESH_SECS = 20

sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import monitor

monitor(STORAGE_ROOT, CONFIG, auto_refresh=AUTO_REFRESH, refresh_secs=REFRESH_SECS)

In [ ]:
# =============================================================================
# Cell 3 — VERIFY  (check artifacts before session ends / for peace of mind)
#
# local storage is persistent — nothing is lost on session expiry.
# This cell is informational: it shows what has been saved.
# =============================================================================

import os, json, pathlib, shutil

_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
STORAGE_HOME = os.path.expanduser("~")

STORAGE_ROOT = f"{STORAGE_HOME}/specdist"

ckpt_dir = os.path.join(STORAGE_ROOT, "checkpoints")
db_path  = os.path.join(STORAGE_ROOT, "results.db")
log_path = os.path.join(STORAGE_ROOT, "logs", "pipeline_output.log")
hf_cache = os.path.join(STORAGE_ROOT, "hf_cache")

print(f"Storage root : {STORAGE_ROOT}")
print()
print(f"results.db   : {os.path.getsize(db_path):,} bytes" if os.path.exists(db_path)
      else "results.db   : NOT FOUND")
print(f"checkpoints/ : {sorted(os.listdir(ckpt_dir)) if os.path.isdir(ckpt_dir) else 'NOT FOUND'}")
print(f"log lines    : {sum(1 for _ in open(log_path))}" if os.path.exists(log_path)
      else "log          : NOT FOUND")

# local storage usage
try:
    total = shutil.disk_usage(STORAGE_HOME)
    used_gb  = (total.total - total.free) / 1024**3
    free_gb  = total.free / 1024**3
    print(f"\nlocal storage    : {used_gb:.1f} GB used, {free_gb:.1f} GB free")
except Exception as e:
    print(f"Disk usage   : could not compute ({e})")

print()
print("All artifacts are on local storage — they persist across sessions automatically.")
print("No manual save needed. Resume with Cell 1 on next session.")

In [ ]:
# =============================================================================
# Cell 4 — PARALLEL TRAINING  (all losses simultaneously)
#
# On AIP with multiple GPUs (up to 3): assigns each loss group to a separate
# GPU so all 15 losses train simultaneously.
#   3 GPUs x 5 losses = ~2-3 h total  (vs 30-40 h sequential on one GPU)
#
# For GPT-2 (FAMILY="gpt2"): launches all losses as CPU background processes.
# No VRAM needed — useful alongside a Qwen GPU run, or on the ATS Cloud server.
#
# Logs go to STORAGE_ROOT/logs/parallel/<family>_<loss>.log
# Results land in the same results.db — dashboard model-family filter
# (DistilGPT-2, LLaMA, Qwen chips) distinguishes pairs automatically.
# =============================================================================

import os, subprocess, sys

# -- Edit these (must match Cell 0) -------------------------------------------
_LDAP        = os.environ.get("USER", "YOUR_LDAP_HERE")
STORAGE_HOME = os.path.expanduser("~")

REPO_DIR     = f"{STORAGE_HOME}/Distill-Spec-Research"
GBV_DIR      = f"{REPO_DIR}/gbv-research"
STORAGE_ROOT = f"{STORAGE_HOME}/specdist"

# FAMILY: which model pair to train
#   "gpt2"  — distilgpt2 (82M) → gpt2-medium (355M)  — CPU, no VRAM
#   "llama" — Llama-3.2-1B → 3B-Instruct              — GPU (needs HF login)
#   "qwen"  — Qwen2.5-0.5B → Qwen3-0.6B/8B            — GPU, use a100/kaggle cfg
FAMILY = "gpt2"     # ← "gpt2" for CPU run; "qwen" for multi-GPU Qwen run

STEPS  = 1000       # steps per loss (gpt2: ~2-4 hr/loss CPU; qwen: ~10-17 min/loss A100)

# GPU assignment for FAMILY="qwen" (ignored for CPU families)
# Adjust group sizes to match your GPU count (AIP gives up to 3)
GPU_GROUPS = {
    0: ["kl", "rev_kl", "jsd", "l1", "kl_tree"],
    1: ["rev_kl_tree", "jsd_tree", "bv_tree", "gbv_tree", "traversal_tree"],
    2: ["naive_tree", "nss_tree", "specinfer_tree", "spectr_tree", "khisti_tree"],
}
# Losses for CPU families (ebe/online excluded per docs/ISSUES.md)
CPU_LOSSES = [
    "kl", "rev_kl", "jsd", "l1",
    "kl_tree", "rev_kl_tree", "jsd_tree",
    "bv_tree", "gbv_tree", "traversal_tree",
    "naive_tree", "nss_tree", "specinfer_tree", "spectr_tree", "khisti_tree",
]
# ---------------------------------------------------------------------------

sys.path.insert(0, GBV_DIR)
from core.model_families import get_family

family_obj = get_family(FAMILY)
draft_id   = family_obj.default_draft_model_id
target_id  = family_obj.default_target_model_id
use_cpu    = FAMILY in ("gpt2",)   # add "llama" here if no GPU

trainer    = os.path.join(GBV_DIR, "algorithms", "distillspec_gbv", "trainer.py")
ckpt_root  = os.path.join(STORAGE_ROOT, "checkpoints")
dataset    = os.path.join(GBV_DIR, "core", "datasets", "raw", "gsm8k_train.jsonl")
log_dir    = os.path.join(STORAGE_ROOT, "logs", "parallel")
os.makedirs(log_dir, exist_ok=True)
os.makedirs(ckpt_root, exist_ok=True)

# Make HF online so models can download if not cached
for _flag in ("TRANSFORMERS_OFFLINE", "HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.pop(_flag, None)

procs = []

if use_cpu:
    # ── CPU path: all losses in parallel ─────────────────────────────────────
    print(f"Launching {len(CPU_LOSSES)} {FAMILY} losses on CPU in parallel...")
    print(f"  draft={draft_id}  target={target_id}  steps={STEPS}")
    for loss in CPU_LOSSES:
        log_path = os.path.join(log_dir, f"{FAMILY}_{loss}.log")
        log_f = open(log_path, "w")
        p = subprocess.Popen(
            [
                sys.executable, trainer,
                "--model_family", FAMILY,
                "--draft",        draft_id,
                "--target",       target_id,
                "--loss",         loss,
                "--steps",        str(STEPS),
                "--device",       "cpu",
                "--no_wandb",
                "--teacher_temp", "1.0",
                "--dataset",      dataset,
                "--output",       os.path.join(ckpt_root, f"{loss}-{FAMILY}"),
            ],
            stdout=log_f, stderr=subprocess.STDOUT, cwd=GBV_DIR,
        )
        procs.append((loss, p, log_f))
        print(f"  [{loss:20s}] PID {p.pid:6d}  log: parallel/{FAMILY}_{loss}.log")

else:
    # ── GPU path: one group per GPU (Qwen / LLaMA with GPU) ──────────────────
    import torch
    n_gpus = torch.cuda.device_count()
    print(f"Detected {n_gpus} GPU(s). Launching loss groups across GPUs...")
    print(f"  draft={draft_id}  target={target_id}  steps={STEPS}")
    for gpu_idx, losses in GPU_GROUPS.items():
        if gpu_idx >= n_gpus:
            print(f"  GPU {gpu_idx}: not available ({n_gpus} GPUs total) — skipping")
            continue
        for loss in losses:
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_idx)}
            log_path = os.path.join(log_dir, f"{FAMILY}_{loss}_gpu{gpu_idx}.log")
            log_f = open(log_path, "w")
            p = subprocess.Popen(
                [
                    sys.executable, trainer,
                    "--model_family", FAMILY,
                    "--draft",        draft_id,
                    "--target",       target_id,
                    "--loss",         loss,
                    "--steps",        str(STEPS),
                    "--device",       "cuda",
                    "--no_wandb",
                    "--dataset",      dataset,
                    "--output",       os.path.join(ckpt_root, f"{loss}-{FAMILY}"),
                ],
                stdout=log_f, stderr=subprocess.STDOUT,
                env=env, cwd=GBV_DIR,
            )
            procs.append((loss, p, log_f))
            print(f"  GPU{gpu_idx} [{loss:20s}] PID {p.pid:6d}")

print(f"\n{len(procs)} jobs running. Waiting (this cell blocks until all finish)...")
print(f"Tail a log: tail -f {os.path.join(log_dir, FAMILY + '_kl.log')}")

failed = []
for loss, p, log_f in procs:
    rc = p.wait()
    log_f.close()
    status = "OK" if rc == 0 else f"FAILED rc={rc}"
    print(f"  [{loss:20s}] {status}")
    if rc != 0:
        failed.append(loss)

if failed:
    print(f"\n{len(failed)} losses failed: {failed}")
    print(f"Tail logs in: {log_dir}")
else:
    print(f"\nAll {len(procs)} losses trained successfully!")
    print("Next: run experiment.py --eval_only or use the eval step in Cell 0/1.")
